# Question 3 - Accelerate Llama2 Inference

Run the cells in order in a Colab GPU runtime. The logs and profiles are in `/content/llama2_q3/results/`.


In [41]:
!nvidia-smi
!nvcc --version
!mkdir -p /content/llama2_q3/results
%cd /content/llama2_q3


Fri Nov 20 01:24:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Write the sources and build

One CUDA thread computes one row of `w @ x`. The naive version allocates device buffers, copies the input and weights, runs the kernel, copies back the result and frees the buffers on every call.

The optimized version uses the same kernel. It copies the checkpoint once and reuses the two device buffers for the input and output vectors. Each call still copies the input vector to the GPU and returns the result to the CPU. [llama2.c](https://github.com/karpathy/llama2.c/tree/350e04fe35433e6d2941dce5a1f53308f87058eb)


In [42]:
%%writefile matmul_cuda.h
#ifndef MATMUL_CUDA_H
#define MATMUL_CUDA_H

#include <stddef.h>

#ifdef __cplusplus
extern "C" {
#endif

void cuda_matmul_naive(float *out, const float *x, const float *w, int n, int d);
void cuda_matmul_init(const void *checkpoint, size_t bytes, int max_n, int max_d);
void cuda_matmul_cached(float *out, const float *x, const float *w, int n, int d);
void cuda_matmul_free(void);

#ifdef __cplusplus
}
#endif

#endif


Overwriting matmul_cuda.h


In [43]:
%%writefile matmul_cuda.cu
#include "matmul_cuda.h"

#include <cuda_runtime.h>
#include <cmath>
#include <cstdio>
#include <cstdlib>

// Used only by the optimized version.
static const float *host_checkpoint;
static float *device_checkpoint;
static float *cached_x;
static float *cached_out;

static bool verify = std::getenv("CHECK_MATMUL") != nullptr;

static void check_cuda(cudaError_t status) {
    if (status != cudaSuccess) {
        std::fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(status));
        std::exit(EXIT_FAILURE);
    }
}

__global__ void matvec_kernel(float *out, const float *x, const float *w, int n, int d) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;
    if (row >= d) {
        return;
    }

    float sum = 0.0f;
    for (int col = 0; col < n; col++) {
        sum += w[(size_t)row * n + col] * x[col];
    }
    out[row] = sum;
}

static void launch_matvec(float *out, const float *x, const float *w, int n, int d) {
    int threads = 128;
    int blocks = (d + threads - 1) / threads;
    matvec_kernel<<<blocks, threads>>>(out, x, w, n, d);
    check_cuda(cudaGetLastError());
}

static void compare_with_cpu(const float *out, const float *x, const float *w, int n, int d) {
    if (!verify) {
        return;
    }

    for (int row = 0; row < d; row++) {
        float expected = 0.0f;
        for (int col = 0; col < n; col++) {
            expected += w[(size_t)row * n + col] * x[col];
        }
        float error = std::fabs(out[row] - expected);
        float tolerance = 1e-5f + 1e-5f * std::fabs(expected);
        if (!(error <= tolerance)) {
            std::fprintf(stderr, "matmul row %d: CPU %g, GPU %g\n", row, expected, out[row]);
            std::exit(EXIT_FAILURE);
        }
    }
}

void cuda_matmul_naive(float *out, const float *x, const float *w, int n, int d) {
    size_t input_bytes = n * sizeof(float);
    size_t output_bytes = d * sizeof(float);
    size_t weight_bytes = input_bytes * d;
    float *device_x;
    float *device_w;
    float *device_out;

    check_cuda(cudaMalloc(&device_w, weight_bytes));
    check_cuda(cudaMalloc(&device_x, input_bytes));
    check_cuda(cudaMalloc(&device_out, output_bytes));
    check_cuda(cudaMemcpy(device_w, w, weight_bytes, cudaMemcpyHostToDevice));
    check_cuda(cudaMemcpy(device_x, x, input_bytes, cudaMemcpyHostToDevice));
    launch_matvec(device_out, device_x, device_w, n, d);
    check_cuda(cudaMemcpy(out, device_out, output_bytes, cudaMemcpyDeviceToHost));
    check_cuda(cudaFree(device_out));
    check_cuda(cudaFree(device_x));
    check_cuda(cudaFree(device_w));
    compare_with_cpu(out, x, w, n, d);
}

void cuda_matmul_init(const void *checkpoint, size_t bytes, int max_n, int max_d) {
    host_checkpoint = static_cast<const float *>(checkpoint);
    check_cuda(cudaMalloc(&device_checkpoint, bytes));
    check_cuda(cudaMalloc(&cached_x, max_n * sizeof(float)));
    check_cuda(cudaMalloc(&cached_out, max_d * sizeof(float)));
    check_cuda(cudaMemcpy(device_checkpoint, checkpoint, bytes, cudaMemcpyHostToDevice));
}

void cuda_matmul_cached(float *out, const float *x, const float *w, int n, int d) {
    // The weights have the same offset in the CPU and GPU checkpoint copies.
    const float *device_w = device_checkpoint + (w - host_checkpoint);
    check_cuda(cudaMemcpy(cached_x, x, n * sizeof(float), cudaMemcpyHostToDevice));
    launch_matvec(cached_out, cached_x, device_w, n, d);
    check_cuda(cudaMemcpy(out, cached_out, d * sizeof(float), cudaMemcpyDeviceToHost));
    compare_with_cpu(out, x, w, n, d);
}

void cuda_matmul_free(void) {
    check_cuda(cudaFree(cached_out));
    check_cuda(cudaFree(cached_x));
    check_cuda(cudaFree(device_checkpoint));
}


Overwriting matmul_cuda.cu


In [44]:
%%writefile run.patch
--- run_upstream.c
+++ run.c
@@ -7,6 +7,9 @@
 #include <math.h>
 #include <string.h>
 #include <fcntl.h>
+#if defined(Q3_CUDA_NAIVE) || defined(Q3_CUDA_CACHED)
+#include "matmul_cuda.h"
+#endif
 #if defined _WIN32
     #include "win.h"
 #else
@@ -166,9 +169,18 @@
     read_checkpoint(checkpoint_path, &t->config, &t->weights, &t->fd, &t->data, &t->file_size);
     // allocate the RunState buffers
     malloc_run_state(&t->state, &t->config);
+#ifdef Q3_CUDA_CACHED
+    Config *p = &t->config;
+    int max_n = p->dim > p->hidden_dim ? p->dim : p->hidden_dim;
+    int max_d = max_n > p->vocab_size ? max_n : p->vocab_size;
+    cuda_matmul_init(t->data, (size_t)t->file_size, max_n, max_d);
+#endif
 }
 
 void free_transformer(Transformer* t) {
+#ifdef Q3_CUDA_CACHED
+    cuda_matmul_free();
+#endif
     // close the memory mapping
     if (t->data != MAP_FAILED) { munmap(t->data, t->file_size); }
     if (t->fd != -1) { close(t->fd); }
@@ -214,7 +226,19 @@
     }
 }
 
+#ifdef Q3_COUNT
+static unsigned long long matmul_calls;
+#endif
+
 void matmul(float* xout, float* x, float* w, int n, int d) {
+#ifdef Q3_COUNT
+    ++matmul_calls;
+#endif
+#if defined(Q3_CUDA_NAIVE)
+    cuda_matmul_naive(xout, x, w, n, d);
+#elif defined(Q3_CUDA_CACHED)
+    cuda_matmul_cached(xout, x, w, n, d);
+#else
     // W (d,n) @ x (n,) -> xout (d,)
     // by far the most amount of time is spent inside this little function
     int i;
@@ -226,9 +250,10 @@
         }
         xout[i] = val;
     }
+#endif
 }
 
 float* forward(Transformer* transformer, int token, int pos) {
 
     // a few convenience variables
     Config* p = &transformer->config;
@@ -968,6 +993,9 @@
     free_sampler(&sampler);
     free_tokenizer(&tokenizer);
     free_transformer(&transformer);
+#ifdef Q3_COUNT
+    fprintf(stderr, "matmul calls: %llu\n", matmul_calls);
+#endif
     return 0;
 }
 #endif


Overwriting run.patch


In [45]:
%%writefile Makefile
CC = gcc
NVCC ?= nvcc
CUDA_ARCH ?= native
UPSTREAM = 350e04fe35433e6d2941dce5a1f53308f87058eb
SOURCE_URL = https://raw.githubusercontent.com/karpathy/llama2.c/$(UPSTREAM)

.DELETE_ON_ERROR:
.PHONY: all

all: run run_count run_naive run_optimized

run_upstream.c:
	curl -fsSL --retry 3 -o $@ $(SOURCE_URL)/run.c

run.c: run_upstream.c run.patch
	patch --batch --output=$@ run_upstream.c < run.patch

run: run_upstream.c
	$(CC) -O3 $< -o $@ -lm

run_count: run.c
	$(CC) -O3 -DQ3_COUNT $< -o $@ -lm

matmul_cuda.o: matmul_cuda.cu matmul_cuda.h
	$(NVCC) -O3 --fmad=false -arch=$(CUDA_ARCH) -c $< -o $@

run_naive.o: run.c matmul_cuda.h
	$(CC) -O3 -DQ3_CUDA_NAIVE -c $< -o $@

run_optimized.o: run.c matmul_cuda.h
	$(CC) -O3 -DQ3_CUDA_CACHED -c $< -o $@

run_naive: run_naive.o matmul_cuda.o
	$(NVCC) -o $@ $^ -lm

run_optimized: run_optimized.o matmul_cuda.o
	$(NVCC) -o $@ $^ -lm


Overwriting Makefile


In [46]:
%%bash
set -e
source_url=https://raw.githubusercontent.com/karpathy/llama2.c/350e04fe35433e6d2941dce5a1f53308f87058eb
model_url=https://huggingface.co/karpathy/tinyllamas/resolve/0bd21da7698eaf29a0d7de3992de8a46ef624add/stories15M.bin

if [ ! -f stories15M.bin ]; then
    curl -fL --retry 3 -o stories15M.bin.part "$model_url?download=true"
    mv stories15M.bin.part stories15M.bin
fi
curl -fsSL -o tokenizer.bin "$source_url/tokenizer.bin"
curl -fsSL -o LICENSE "$source_url/LICENSE"
make


patch --batch --output=run.c run_upstream.c < run.patch
patching file run.c (read from run_upstream.c)
gcc -O3 -DQ3_COUNT run.c -o run_count -lm
gcc -O3 -DQ3_CUDA_NAIVE -c run.c -o run_naive.o
nvcc -O3 --fmad=false -arch=native -c matmul_cuda.cu -o matmul_cuda.o
nvcc -o run_naive run_naive.o matmul_cuda.o -lm
gcc -O3 -DQ3_CUDA_CACHED -c run.c -o run_optimized.o
nvcc -o run_optimized run_optimized.o matmul_cuda.o -lm


## 2. Run the CPU version and count matmul calls

The first run uses the README command and the original FP32 CPU code. The comparison run uses an empty prompt and the flags `-t 0 -s 42 -n 256`, just like the two CUDA runs.

`run_count` counts calls to `matmul`. Each forward pass makes seven calls per layer and one for the vocabulary output, so the six-layer model makes 43 calls per pass.


In [47]:
%%bash
set -euo pipefail
echo '$ ./run stories15M.bin'
./run stories15M.bin 2>&1 | tee results/cpu_default.txt


$ ./run stories15M.bin
Once upon a time, there was a little girl named Lily. She loved to play outside in the park. One day, Lily saw a frog jumping around. The frog was green and jumped around like a ball.
Lily was a little bit nervous because she had never seen a frog before. But the frog was friendly and wanted to be her friend. So, Lily and the frog played together in the park.
They jumped and jumped all day long. Lily was so happy to have a new friend. She felt like she had found something special. When it was time to go home, Lily said goodbye to the frog and promised to come back and play again.achieved tok/s: 43.416167



In [48]:
%%bash
set -e
echo '$ ./run stories15M.bin -t 0 -s 42 -n 256'
./run stories15M.bin -t 0 -s 42 -n 256 > results/cpu.txt 2> results/cpu_timing.txt
cat results/cpu.txt results/cpu_timing.txt

./run_count stories15M.bin -t 0 -s 42 -n 256 > /dev/null 2> results/count.txt
grep '^matmul calls:' results/count.txt


$ ./run stories15M.bin -t 0 -s 42 -n 256
Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big, red ball in the sky. It was the sun! She thought it was so pretty.
Lily wanted to play with the ball, but it was too high up in the sky. She tried to jump and reach it, but she couldn't. Then, she had an idea. She would use a stick to knock the ball down.
Lily found a stick and tried to hit the ball. But the stick was too short. She tried again and again, but she couldn't reach it. She felt sad.
Suddenly, a kind man came by and saw Lily. He asked her what was wrong. Lily told him about the ball. The man smiled and said, "I have a useful idea!" He took out a long stick and used it to knock the ball down. Lily was so happy! She thanked the man and they played together in the sunshine.
achieved tok/s: 45.926850
matmul calls: 9546


## 3. Run naive CUDA

In [49]:
%%bash
set -e
echo '$ ./run_naive stories15M.bin -t 0 -s 42 -n 256'
./run_naive stories15M.bin -t 0 -s 42 -n 256 > results/naive.txt 2> results/naive_timing.txt
cat results/naive.txt results/naive_timing.txt


$ ./run_naive stories15M.bin -t 0 -s 42 -n 256
Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big, red ball in the sky. It was the sun! She thought it was so pretty.
Lily wanted to play with the ball, but it was too high up in the sky. She tried to jump and reach it, but she couldn't. Then, she had an idea. She would use a stick to knock the ball down.
Lily found a stick and tried to hit the ball. But the stick was too short. She tried again and again, but she couldn't reach it. She felt sad.
Suddenly, a kind man came by and saw Lily. He asked her what was wrong. Lily told him about the ball. The man smiled and said, "I have a useful idea!" He took out a long stick and used it to knock the ball down. Lily was so happy! She thanked the man and they played together in the sunshine.
achieved tok/s: 34.661230


## 4. Run optimized CUDA

In [50]:
%%bash
set -e
echo '$ ./run_optimized stories15M.bin -t 0 -s 42 -n 256'
./run_optimized stories15M.bin -t 0 -s 42 -n 256 > results/optimized.txt 2> results/optimized_timing.txt
cat results/optimized.txt results/optimized_timing.txt


$ ./run_optimized stories15M.bin -t 0 -s 42 -n 256
Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big, red ball in the sky. It was the sun! She thought it was so pretty.
Lily wanted to play with the ball, but it was too high up in the sky. She tried to jump and reach it, but she couldn't. Then, she had an idea. She would use a stick to knock the ball down.
Lily found a stick and tried to hit the ball. But the stick was too short. She tried again and again, but she couldn't reach it. She felt sad.
Suddenly, a kind man came by and saw Lily. He asked her what was wrong. Lily told him about the ball. The man smiled and said, "I have a useful idea!" He took out a long stick and used it to knock the ball down. Lily was so happy! She thanked the man and they played together in the sunshine.
achieved tok/s: 283.697047


In [51]:
!grep -H "achieved tok/s" results/*_timing.txt


results/cpu_timing.txt:achieved tok/s: 45.926850
results/naive_timing.txt:achieved tok/s: 34.661230
results/optimized_timing.txt:achieved tok/s: 283.697047


## 5. Check correctness

Use `diff` to compare the generated text. With `CHECK_MATMUL=1`, both CUDA versions also compare each output element with the CPU result during a short run.


In [52]:
%%bash
set -e
diff results/cpu.txt results/naive.txt
diff results/cpu.txt results/optimized.txt
echo "The three runs generated the same text."

CHECK_MATMUL=1 ./run_naive stories15M.bin -t 0 -s 42 -n 8 > /dev/null
CHECK_MATMUL=1 ./run_optimized stories15M.bin -t 0 -s 42 -n 8 > /dev/null
echo "Both CUDA versions passed the CPU matmul check."


The three runs generated the same text.
Both CUDA versions passed the CPU matmul check.


achieved tok/s: 20.588235
achieved tok/s: 41.666667


## 6. Profile naive and optimized CUDA

Use the same input and options. The traces include CUDA calls, kernel launches and memory transfers. If Colab already has `nsys`, use it. Otherwise extract the NVIDIA package below.


In [53]:
from pathlib import Path
import os
import shutil
import subprocess

nsys = shutil.which("nsys")
if nsys is None:
    url = ("https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/"
           "nsight-systems-2025.6.3_2025.6.3.541-1_amd64.deb")
    subprocess.run(["curl", "-fL", "--retry", "3", "-o", "nsight.deb", url], check=True)
    subprocess.run(["dpkg-deb", "-x", "nsight.deb", "nsight"], check=True)
    nsys = str(next(Path("nsight").rglob("target-linux-x64/nsys")).resolve())

os.environ["NSYS"] = nsys
!{nsys} --version


NVIDIA Nsight Systems version 2025.3.1.0


In [54]:
%%bash
set -euo pipefail
"$NSYS" profile --trace=cuda --sample=none --cpuctxsw=none --force-overwrite=true \
    -o results/naive ./run_naive stories15M.bin -t 0 -s 42 -n 256
"$NSYS" stats --force-export=true \
    --report cuda_api_sum,cuda_gpu_kern_sum,cuda_gpu_mem_time_sum,cuda_gpu_mem_size_sum \
    results/naive.nsys-rep | tee results/naive_stats.txt


Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big, red ball in the sky. It was the sun! She thought it was so pretty.
Lily wanted to play with the ball, but it was too high up in the sky. She tried to jump and reach it, but she couldn't. Then, she had an idea. She would use a stick to knock the ball down.
Lily found a stick and tried to hit the ball. But the stick was too short. She tried again and again, but she couldn't reach it. She felt sad.
Suddenly, a kind man came by and saw Lily. He asked her what was wrong. Lily told him about the ball. The man smiled and said, "I have a useful idea!" He took out a long stick and used it to knock the ball down. Lily was so happy! She thanked the man and they played together in the sunshine.
Generating '/tmp/nsys-report-a1a0.qdstrm'
[1/1] [========================100%] naive.nsys-rep
Generated:
	/content/llama2_q3/results/naive.nsys-rep
Generating SQLite file results/naive.sq

achieved tok/s: 31.807714


In [55]:
%%bash
set -euo pipefail
"$NSYS" profile --trace=cuda --sample=none --cpuctxsw=none --force-overwrite=true \
    -o results/optimized ./run_optimized stories15M.bin -t 0 -s 42 -n 256
"$NSYS" stats --force-export=true \
    --report cuda_api_sum,cuda_gpu_kern_sum,cuda_gpu_mem_time_sum,cuda_gpu_mem_size_sum \
    results/optimized.nsys-rep | tee results/optimized_stats.txt


Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big, red ball in the sky. It was the sun! She thought it was so pretty.
Lily wanted to play with the ball, but it was too high up in the sky. She tried to jump and reach it, but she couldn't. Then, she had an idea. She would use a stick to knock the ball down.
Lily found a stick and tried to hit the ball. But the stick was too short. She tried again and again, but she couldn't reach it. She felt sad.
Suddenly, a kind man came by and saw Lily. He asked her what was wrong. Lily told him about the ball. The man smiled and said, "I have a useful idea!" He took out a long stick and used it to knock the ball down. Lily was so happy! She thanked the man and they played together in the sunshine.
Generating '/tmp/nsys-report-450b.qdstrm'
[1/1] [========================100%] optimized.nsys-rep
Generated:
	/content/llama2_q3/results/optimized.nsys-rep
Generating SQLite file results/

achieved tok/s: 247.203579


### Compare the two profiles

Compare allocation and copy calls in the CUDA API Summary, copy sizes and times in the memory summaries, and kernel launches in the kernel summary. Use the unprofiled runs for `achieved tok/s`. [Nsight Systems report guide](https://docs.nvidia.com/nsight-systems/AnalysisGuide/index.html)


## 7. Download the source files

This ZIP contains the C and CUDA sources, Makefile, patch and license. The timing logs and Nsight profiles are saved under `results/`.


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

sources = ["run.c", "run_upstream.c", "matmul_cuda.cu", "matmul_cuda.h",
           "Makefile", "run.patch", "LICENSE"]

with ZipFile("llama2_q3_source.zip", "w", ZIP_DEFLATED) as archive:
    for name in sources:
        archive.write(name)

files.download("llama2_q3_source.zip")
